# Sales Intelligence Research Automation - Demo

This notebook is a **self-contained demo** of a CrewAI crew that automates sales intelligence research.
Given a prospect's email address, the crew runs four agents sequentially:

| # | Agent | Data Source |
|---|-------|-------------|
| 1 | Company Research Specialist | Serper (web search) |
| 2 | Email History Analyst | Gmail API |
| 3 | Sales Intelligence Report Writer | Synthesizes all of the above |



> **Prerequisites:** API keys for OpenAI, Serper, and NVIDIA NIM.  
> The HubSpot and Gmail integrations use [CrewAI Apps](https://docs.crewai.com/en/enterprise/features/tools-and-integrations) and require platform authentication.

In [ ]:
!pip install 'crewai[litellm,tools]'
!pip install 'python-dotenv'


## Setup

Set your API keys below. The `getpass` prompts ensure secrets are never stored in the notebook.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

if not os.environ.get("SERPER_API_KEY"):
    os.environ["SERPER_API_KEY"] = getpass("Serper API Key: ")

if not os.environ.get("NVIDIA_NIM_API_KEY"):
    os.environ["NVIDIA_NIM_API_KEY"] = getpass("NVIDIA NIM API Key: ")


In [ ]:
from crewai import Agent, Crew, Process, Task, LLM
from crewai_tools import SerperDevTool

nvidia_llm = LLM(model="nvidia_nim/nvidia/nemotron-3-super-120b-a12b", temperature=0.7)
openai_llm = LLM(model="openai/gpt-4o-mini", temperature=0.7)


## Define Agents

Four specialized agents, each with a distinct role in the sales intelligence pipeline.
The `{prospect_email}` placeholders are interpolated at runtime by `crew.kickoff(inputs=...)`.

In [ ]:
company_research_specialist = Agent(
    role="Company Research Specialist",
    goal=(
        "Research {prospect_email} and their company thoroughly to understand their "
        "business, industry, recent developments, and potential pain points that our "
        "solutions could address"
    ),
    backstory=(
        "You are an expert business intelligence researcher with years of experience "
        "analyzing companies and their key personnel. You excel at finding publicly "
        "available information about businesses, their challenges, growth opportunities, "
        "and industry trends. You understand how to identify decision-makers and their "
        "priorities through online research."
    ),
    tools=[SerperDevTool()],
    inject_date=True,
    allow_delegation=False,
    max_iter=25,
    verbose=True,
    llm=nvidia_llm,
)


email_history_analyst = Agent(
    role="Email History Analyst",
    goal=(
        "Search through Gmail to find and analyze any previous email exchanges with "
        "{prospect_email}, understanding the communication history, tone, and context "
        "of past interactions"
    ),
    backstory=(
        "You are an email communication specialist who excels at analyzing email "
        "patterns and conversation history. You understand how to extract meaningful "
        "insights from email exchanges, including communication preferences, "
        "relationship status, and previous discussion topics that can inform future "
        "sales conversations."
    ),
    apps=["google_gmail/fetch_emails"],
    inject_date=True,
    allow_delegation=False,
    max_iter=25,
    verbose=True,
    llm=openai_llm,
)

sales_intelligence_report_writer = Agent(
    role="Sales Intelligence Report Writer",
    goal=(
        "Synthesize all research findings from company research, HubSpot data, and "
        "email history to create a comprehensive sales intelligence report that "
        "prepares the sales team for an effective initial sales call with "
        "{prospect_email}"
    ),
    backstory=(
        "You are a senior sales strategist with expertise in preparing sales teams "
        "for high-impact prospect meetings. You know how to synthesize diverse data "
        "sources into actionable sales intelligence, identifying key talking points, "
        "potential objections, and strategic approaches. Your reports are known for "
        "being concise yet comprehensive, giving sales teams the confidence and "
        "knowledge they need to succeed."
    ),
    inject_date=True,
    allow_delegation=False,
    max_iter=25,
    verbose=True,
    llm=nvidia_llm,
)


## Define Tasks

Each task maps to one agent. The final report task receives `context` from the first three,
so it can synthesize all findings.

In [ ]:
research_task = Task(
    description=(
        "Conduct comprehensive research on {prospect_email} to identify the person "
        "and their company. Research their business model, industry position, recent "
        "news, key personnel, company size, funding status, and potential business "
        "challenges or opportunities. Look for any public information about their "
        "role, responsibilities, and professional background."
    ),
    expected_output=(
        "A detailed research report covering the prospect's professional profile, "
        "company overview, industry analysis, recent developments, key challenges, "
        "and potential business opportunities that could be relevant for our sales "
        "approach."
    ),
    agent=company_research_specialist,
)

email_task = Task(
    description=(
        "Search through Gmail for any previous email exchanges with "
        "{prospect_email}. Analyze the communication patterns, frequency, tone, "
        "topics discussed, and overall relationship development. Look for any "
        "mentions of business needs, pain points, projects, or interests that "
        "were discussed in previous emails."
    ),
    expected_output=(
        "A detailed analysis of email communication history with the prospect, "
        "including conversation themes, relationship warmth, communication "
        "preferences, any discussed business needs or challenges, and "
        "recommendations for conversation approach based on past interactions. "
        "If no emails exist, confirm this finding."
    ),
    agent=email_history_analyst,
)

report_task = Task(
    description=(
        "Synthesize all research findings from company research, HubSpot data "
        "analysis, and email history to create a comprehensive sales intelligence "
        "report. The report should include prospect profile, company overview, "
        "relationship context, potential pain points, business opportunities, "
        "conversation starters, recommended approach, and key questions to ask "
        "during the initial sales call."
    ),
    expected_output=(
        "A professional sales intelligence report in markdown format that includes: "
        "Executive Summary, Prospect Profile, Company Analysis, Relationship History, "
        "Key Insights, Recommended Sales Approach, Conversation Starters, Potential "
        "Objections & Responses, and Next Steps. The report should be actionable and "
        "ready to use for the sales call."
    ),
    agent=sales_intelligence_report_writer,
    context=[research_task, email_task],
)


## Run the Crew

Change `prospect_email` below to research a different prospect.

In [20]:
prospect_email = "fake_email@crewai.com"

crew = Crew(
    agents=[
        company_research_specialist,
        hubspot_data_analyst,
        email_history_analyst,
        sales_intelligence_report_writer,
    ],
    tasks=[research_task, email_task, report_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff(inputs={"prospect_email": prospect_email})


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0f01862f-db07-4b8a-b78a-2a7b87b139ac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: c171d554-33c5-4468-883c-18a80ede98c6
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: c171d554-33c5-4468-883c-18a80ede98c6
       Status: Executing Task...
    └── 🤖 Agent: Company Research Specialist
            Status: In Progress

# Agent: Company Research Specialist
## Task: Conduct comprehensive research on fake_email@crewai.com to identify the person and their company. Research their business model, industry position, recent news, key personnel, company size, funding status, and potential business challenges or opportunities. Look for any public information about their role, responsibilities, and professional background.


# Agent: Company Research Specialist
## Final Answer: 
{
  "Thought": "I need to search for the email address fake_email@crewai.com to see if any public information exists about the person and their company.",
  "Action": "Search the internet with Serper",
  "Action Input": {
    "search_query": "\"fake_email@crewai.com\""
  }
}




🚀 Crew: crew
└── 📋 Task: c171d554-33c5-4468-883c-18a80ede98c6
       Status: Executing Task...
    └── 🤖 Agent: Company Research Specialist
            Status: ✅ Completed

...

# Agent: Sales Intelligence Report Writer
## Task: Synthesize all research findings from company research, HubSpot data analysis, and email history to create a comprehensive sales intelligence report. The report should include prospect profile, company overview, relationship context, potential pain points, business opportunities, conversation starters, recommended approach, and key questions to ask during the initial sales call.


# Agent: Sales Intelligence Report Writer
## Final Answer: 
{
  "Thought": "I need to gather information about the company crewai.com to fill the prospect profile and company analysis sections.",
  "Action": "Search the internet with Serper",
  "Action Input": {
    "search_query": "crewai.com"
  }
}




🚀 Crew: crew
├── 📋 Task: c171d554-33c5-4468-883c-18a80ede98c6
│      Assigned to: Company Research Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Company Research Specialist
│           Status: ✅ Completed
├── 📋 Task: e435a947-7a9a-47e4-8277-673250471551
│      Assigned to: Email History Analyst
│      Status: ✅ Completed
│   └── 🤖 Agent: Email History Analyst
│           Status: ✅ Completed
└── 📋 Task: 8dc4a875-6cd0-459a-9254-66b88feff849
       Status: Executing Task...
    └── 🤖 Agent: Sales Intelligence Report Writer
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: c171d554-33c5-4468-883c-18a80ede98c6
│      Assigned to: Company Research Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Company Research Specialist
│           Status: ✅ Completed
├── 📋 Task: e435a947-7a9a-47e4-8277-673250471551
│      Assigned to: Email History Analyst
│      Status: ✅ Completed
│   └── 🤖 Agent: Email History Analyst
│           Status: ✅ Completed
└── 📋 Task: 8dc4a875-6cd0-459a-9254-66b88feff849
       Assigned to: Sales Intelligence Report Writer
       Status: ✅ Completed
    └── 🤖 Agent: Sales Intelligence Report Writer
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 8dc4a875-6cd0-459a-9254-66b88feff849                                                                     │
│  Agent: Sales Intelligence Report Writer                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0f01862f-db07-4b8a-b78a-2a7b87b139ac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [21]:
from IPython.display import Markdown, display

display(Markdown(result.raw))


{
  "Thought": "I need to gather information about the company crewai.com to fill the prospect profile and company analysis sections.",
  "Action": "Search the internet with Serper",
  "Action Input": {
    "search_query": "crewai.com"
  }
}